In [ ]:
import sys, platform, subprocess
print(sys.version); print(platform.libc_ver())
print(subprocess.run([sys.executable, '-m', 'pip', '--version'], capture_output=True, text=True).stdout)

In [ ]:
! mkdir -p /kaggle/working/wheels /kaggle/working/sdist
# 1. SGLang + deps (binary wheels where they exist; sdists land in sdist/ and are built below)
! pip download --dest /kaggle/working/wheels --prefer-binary sglang==0.5.20 2>&1 | tail -n 25

In [ ]:
# 2. Build sdists into wheels (cuda-tile 1.6.0rc5 is sdist-only)
import glob, os, subprocess, sys, shutil
for f in glob.glob('/kaggle/working/wheels/*.tar.gz') + glob.glob('/kaggle/working/wheels/*.zip'):
    print('building', os.path.basename(f))
    r = subprocess.run([sys.executable, '-m', 'pip', 'wheel', '--no-deps', '-w', '/kaggle/working/wheels', f],
                       capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
    shutil.move(f, '/kaggle/working/sdist/')
print(len(glob.glob('/kaggle/working/wheels/*.whl')), 'wheels')

In [ ]:
# 3. FlashInfer cubins + JIT cache for CUDA 13.0 (avoids runtime downloads / nvcc JIT offline)
! pip download --dest /kaggle/working/wheels --no-deps --find-links https://flashinfer.ai/whl/flashinfer-cubin/ flashinfer-cubin==0.6.18 2>&1 | tail -n 3
! pip download --dest /kaggle/working/wheels --no-deps --index-url https://flashinfer.ai/whl/cu130 flashinfer-jit-cache==0.6.18 2>&1 | tail -n 3

In [ ]:
# 4. Offline resolve check against the wheel dir only + lock file + hashes
import subprocess, sys, json, hashlib, glob, os
W = '/kaggle/working/wheels'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--dry-run', '--ignore-installed', '--no-index', '--find-links', W,
                    '--report', '/kaggle/working/resolve-report.json', 'sglang==0.5.20', 'flashinfer-cubin==0.6.18',
                    'flashinfer-jit-cache==0.6.18'], capture_output=True, text=True)
print('offline resolve rc', r.returncode); print(r.stdout[-1500:]); print(r.stderr[-1500:])
rep = json.load(open('/kaggle/working/resolve-report.json'))
lock = sorted(f"{i['metadata']['name']}=={i['metadata']['version']}" for i in rep['install'])
open(f'{W}/requirements.lock', 'w').write('\n'.join(lock) + '\n')
sums, total = [], 0
for f in sorted(glob.glob(W + '/*.whl')):
    h = hashlib.sha256()
    with open(f, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 24), b''):
            h.update(b)
    sums.append(f'{h.hexdigest()}  {os.path.basename(f)}'); total += os.path.getsize(f)
open(f'{W}/SHA256SUMS', 'w').write('\n'.join(sums) + '\n')
json.dump({'packages': 'sglang==0.5.20', 'flashinfer': '0.6.18', 'n_wheels': len(sums), 'bytes': total,
           'python': sys.version}, open(f'{W}/manifest.json', 'w'), indent=1)
print(len(lock), 'packages in lock;', len(sums), 'wheels;', round(total / 1e9, 2), 'GB')
print([l for l in lock if l.split('==')[0] in ('sglang', 'sglang-kernel', 'torch', 'flashinfer-python', 'flashinfer-cubin',
       'flashinfer-jit-cache', 'transformers', 'triton', 'cuda-tile', 'xgrammar')])